# Time-Resolved XRD Analysis

Load compact processed 1-D stacks into xarray, keep raw/cake rows lazy,
normalize and bin with provenance, inspect a pilot fit, then derive a
lattice and explicitly illustrative temperature history. Physical rates
require explicit seconds and a calibration.


In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import (
    LinearThermalExpansion, PeakFitPlan, add_lattice_results,
    add_temperature_results, bin_time_resolved, fit_peak_series,
    export_time_resolved_results,
    discover_processed_scans, flag_fit_quality, flag_normalization_outliers, load_time_resolved_series,
    normalize_monitor, normalize_reference_band, select_time_zero,
)
from xrd_tools.gui.widgets import ImageViewer, PatternViewer, PeakFitControls
from xrd_tools.core import IntegrationResult1D, IntegrationResult2D
from xrd_tools.io.nexus import write_nexus
from xrd_tools.viz import plot_peak_fit_frame, plot_thermal_history, plot_time_resolved_waterfall


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
processed_root = TEST_DATA / "xdart_processed_data"
processed_file = widgets.Text(value=str(processed_root / "Pt_test_burst_00007.nexus"), description="processed file")
processed_folder = widgets.Text(value=str(processed_root), description="processed folder")
selection_mode = widgets.ToggleButtons(options=("file", "folder"), value="file", description="load")
scan_filter = widgets.Text(value="*.nexus", description="scan filter")
discover_button = widgets.Button(description="Find processed scans")
scan_selection = widgets.SelectMultiple(options=(), description="selected scans", layout=widgets.Layout(width="620px", height="120px"))
raw_root = widgets.Text(value=str(TEST_DATA) if TEST_DATA.exists() else "", description="raw root")
time_key = widgets.Text(value="", description="time key")
time_unit = widgets.Dropdown(options=("s", "ms", "us"), value="s", description="time unit")
frame_period_ms = widgets.BoundedFloatText(value=2.0, min=0.001, max=1e6, step=0.1, description="period ms")
monitor_method = widgets.Dropdown(options=("reference band", "monitor"), value="reference band", description="normalize")
monitor_key = widgets.Text(value="i0", description="monitor key")
q_band = widgets.FloatRangeSlider(value=(3.05, 3.20), min=2.4, max=3.3, step=0.01, description="reference q", continuous_update=False)
bin_size = widgets.BoundedIntText(value=2, min=1, max=100, description="bin size")
frame_row = widgets.BoundedIntText(value=0, min=0, max=0, description="row")
batch_limit = widgets.BoundedIntText(value=8, min=1, max=1000, description="batch rows")
thermal_time_basis = widgets.Dropdown(options=("automatic", "time", "sequence_time"), value="automatic", description="thermal time")
load_button = widgets.Button(description="Load processed", button_style="primary")
preprocess_button = widgets.Button(description="Apply preprocessing")
inspect_button = widgets.Button(description="Inspect lazy row")
pilot_button = widgets.Button(description="Run pilot fit", button_style="primary")
batch_button = widgets.Button(description="Run bounded batch", button_style="primary")
export_button = widgets.Button(description="Export compact results")
export_directory = Path(os.environ.get("XDART_NOTEBOOK_OUTPUT", tempfile.gettempdir()))
status = widgets.HTML("<i>Load, preprocess, and fit actions are explicit; row selection only reads cached/lazy data.</i>")
output = widgets.Output()
display(widgets.VBox([selection_mode, processed_file, processed_folder, widgets.HBox([scan_filter, discover_button]), scan_selection, raw_root, widgets.HBox([time_key, time_unit, frame_period_ms]), widgets.HBox([monitor_method, monitor_key]), q_band, widgets.HBox([bin_size, frame_row, batch_limit, thermal_time_basis]), widgets.HBox([load_button, preprocess_button, inspect_button]), widgets.HBox([pilot_button, batch_button, export_button]), status, output]))


In [ ]:
NOTEBOOK_STATE = {"loads": 0, "preprocesses": 0, "series": None, "prepared": None, "fits": None, "thermal": None, "raw_reads": 0, "discovered_scans": (), "selected_scans": (), "thermal_time_coord": None}
if SMOKE_MODE:
    smoke_directory = tempfile.TemporaryDirectory(prefix="xdart_notebook_time_resolved_")

def _smoke_file():
    path = Path(smoke_directory.name) / "scan.nexus"
    q, frames = np.linspace(2.45, 3.30, 240), np.arange(16, dtype=np.int64)
    chi = np.linspace(-30, 30, 12)
    centers = 2.765 - 0.0008 * frames
    stack = np.array([15 + 180 * np.exp(-0.5 * ((q - center) / 0.018) ** 2) for center in centers], dtype=np.float32)
    write_nexus(
        path,
        results_1d={int(frame): IntegrationResult1D(q, row, sigma=np.sqrt(row), unit="q_A^-1") for frame, row in zip(frames, stack)},
        results_2d={int(frame): IntegrationResult2D(q, chi, np.broadcast_to(row[:, None], (q.size, chi.size)), unit="q_A^-1", azimuthal_unit="chi_deg") for frame, row in zip(frames, stack)},
        overwrite=True,
        compression=None,
    )
    with h5py.File(path, "r+") as h5:
        entry = h5["entry"]
        scan_data = entry.create_group("scan_data"); scan_data.create_dataset("frame_index", data=frames)
        scan_data.create_dataset("elapsed", data=frames * 2.0); scan_data.create_dataset("i0", data=np.linspace(0.98, 1.02, len(frames)))
        groups = entry.create_group("frames")
        for frame in frames:
            groups.create_group(f"frame_{frame:04d}").create_dataset("thumbnail", data=np.ones((8, 8), dtype=np.uint8))
    return path

def discover_folder_scans(_=None):
    with output:
        clear_output(wait=True)
        try:
            assert selection_mode.value == "folder", "Choose folder mode before discovering scans"
            root = Path(processed_folder.value).expanduser()
            pattern = scan_filter.value.strip() or "*.nexus"
            paths = discover_processed_scans(root, pattern=pattern)
            assert paths, f"No processed 1-D scans match {pattern!r} under {root}"
            scan_selection.options = [(path.name, str(path)) for path in paths]
            scan_selection.value = ()
            NOTEBOOK_STATE["discovered_scans"] = tuple(paths)
            display({"discovered": [path.name for path in paths], "next": "select one or more scans, then load"})
            status.value = f"<b>Discovered {len(paths)} naturally sorted processed scans; select the subset to load.</b>"
        except Exception as exc:
            status.value = f"<b>Scan discovery failed:</b> {exc}"
            raise

def _selected_paths():
    if SMOKE_MODE:
        return _smoke_file()
    if selection_mode.value == "file":
        candidate = Path(processed_file.value).expanduser()
        assert candidate.is_file(), f"Missing processed selection: {candidate}"
        return candidate
    paths = tuple(Path(value) for value in scan_selection.value)
    assert paths, "Discover a folder and select one or more processed scans before loading"
    discovered = set(NOTEBOOK_STATE["discovered_scans"])
    assert set(paths).issubset(discovered), "Selected scans are not from the current discovered folder"
    return paths

def load_processed(_=None):
    with output:
        clear_output(wait=True)
        try:
            selected_key = "elapsed" if SMOKE_MODE else time_key.value.strip() or None
            selected_unit = "ms" if SMOKE_MODE else (time_unit.value if selected_key else None)
            paths = _selected_paths()
            selected_names = [paths.name] if isinstance(paths, Path) else [path.name for path in paths]
            series = load_time_resolved_series(paths, frame_period_s=frame_period_ms.value / 1000.0, time_key=selected_key, time_unit=selected_unit, metadata_keys=(monitor_key.value.strip(),) if monitor_key.value.strip() else (), source_root=raw_root.value.strip() or None)
            frame_row.max, frame_row.value = series.dataset.sizes["pattern"] - 1, 0
            NOTEBOOK_STATE.update(loads=NOTEBOOK_STATE["loads"] + 1, series=series, prepared=None, fits=None, thermal=None, selected_scans=tuple(selected_names), thermal_time_coord=None)
            status.value = f"<b>Loaded {series.dataset.sizes['pattern']} scan-qualified patterns from {len(selected_names)} selected scan(s).</b>"
            display({"selected_scans": selected_names, "patterns": series.dataset.sizes["pattern"], "q_points": series.dataset.sizes["q"], "time_units": series.dataset.coords["time"].attrs["units"]})
        except Exception as exc:
            status.value = f"<b>Load failed:</b> {exc}"
            raise

def apply_preprocessing(_=None):
    with output:
        clear_output(wait=True)
        try:
            series = NOTEBOOK_STATE["series"]
            assert series is not None, "Load processed data first"
            dataset, fallback = series.dataset, False
            if monitor_method.value == "monitor":
                try:
                    candidate = normalize_monitor(dataset, monitor_key.value.strip())
                    if bool(candidate["monitor_normalization_valid"].any()):
                        dataset, source_var = candidate, "intensity_normalized"
                    else:
                        fallback, source_var = True, "intensity"
                except (KeyError, ValueError):
                    fallback, source_var = True, "intensity"
            else:
                source_var = "intensity"
            normalized = normalize_reference_band(dataset, q_range=tuple(q_band.value), intensity_var=source_var, output_var="intensity_band_normalized")
            prepared = select_time_zero(bin_time_resolved(flag_normalization_outliers(normalized), bin_size=bin_size.value), zero_pattern=0)
            NOTEBOOK_STATE.update(preprocesses=NOTEBOOK_STATE["preprocesses"] + 1, prepared=prepared, fits=None, thermal=None)
            display(plot_time_resolved_waterfall(prepared, intensity_var="intensity_band_normalized", log_intensity=True))
            status.value = "<b>Reference-band preprocessing complete.</b>" if not fallback else "<b>Selected monitor was unavailable; used documented reference-band fallback.</b>"
        except Exception as exc:
            status.value = f"<b>Preprocessing failed:</b> {exc}"
            raise

def inspect_lazy_row(_=None):
    with output:
        clear_output(wait=True)
        try:
            series = NOTEBOOK_STATE["series"]
            assert series is not None, "Load processed data first"
            row = int(frame_row.value)
            thumbnail, cake = series.get_thumbnail(row), series.get_cake(row)
            try:
                image, title, raw_available = series.get_raw(row), "Full raw detector frame", True
            except (KeyError, FileNotFoundError, ValueError):
                image, title, raw_available = thumbnail, "Stored thumbnail (raw unavailable)", False
            pattern = series.dataset.isel(pattern=row)
            display(widgets.HBox([PatternViewer(patterns=[(pattern.q.values, pattern.intensity.values, f"row {row}")]).widget, ImageViewer(image, title=title).widget]))
            display({"scan": str(pattern.scan_name.item()), "frame_label": int(pattern.frame_label.item()), "cake_shape": cake.intensity.shape, "raw_available": raw_available})
            NOTEBOOK_STATE["raw_reads"] += 1
        except Exception as exc:
            status.value = f"<b>Lazy inspection failed:</b> {exc}"
            raise

def _plan_from_controls(params):
    positions = tuple(params["positions"] or (2.76,))
    return PeakFitPlan(positions=positions, model=params["model"], background=params["background"] if params["background"] in {"none", "constant", "linear"} else "linear", sigma_init=params["sigma_init"], sigma_bounds=params["sigma_bounds"], center_bounds_delta=params["center_bounds_delta"], fit_kwargs={"method": "leastsq"})

def run_pilot(params=None):
    with output:
        clear_output(wait=True)
        try:
            prepared = NOTEBOOK_STATE["prepared"]
            assert prepared is not None, "Apply preprocessing first"
            fits = fit_peak_series(prepared, _plan_from_controls(params or fit_controls.get_params()), intensity_var="intensity_band_normalized", q_range=(fit_controls.q_min.value, fit_controls.q_max.value), pattern_indices=[min(int(frame_row.value), prepared.sizes["pattern"] - 1)])
            NOTEBOOK_STATE["fits"] = flag_fit_quality(fits, max_center_error=0.02)
            display(plot_peak_fit_frame(NOTEBOOK_STATE["fits"], 0))
            status.value = "<b>Pilot fit complete.</b>"
        except Exception as exc:
            status.value = f"<b>Pilot fit failed:</b> {exc}"
            raise

fit_controls = PeakFitControls(on_fit=run_pilot)
peak_center = 2.76 if SMOKE_MODE else 1.56
fit_controls.n_peaks.value = 1; fit_controls.peak_positions.value = str(peak_center); fit_controls.peak_model.value = "gaussian"; fit_controls.bg_model.value = "linear"; fit_controls.sigma_init.value = 0.02; fit_controls.q_min.value, fit_controls.q_max.value = peak_center - 0.16, peak_center + 0.16
display(fit_controls.widget)

def _thermal_time_coordinate(dataset):
    scan_count = len(np.unique(dataset.coords["scan_index"].values))
    requested = thermal_time_basis.value
    coordinate = requested if requested != "automatic" else ("time" if scan_count == 1 else "sequence_time")
    values = np.asarray(dataset.coords[coordinate].values, dtype=float)
    units = str(dataset.coords[coordinate].attrs.get("units", ""))
    if units not in {"s", "second", "seconds"} or not np.all(np.isfinite(values)) or np.any(np.diff(values) <= 0):
        raise ValueError(
            f"{coordinate!r} is not a finite strictly increasing physical-seconds coordinate; "
            "select one scan or provide proven cross-scan cadence before deriving K/s"
        )
    return coordinate

def run_bounded_batch(_=None):
    with output:
        clear_output(wait=True)
        try:
            prepared = NOTEBOOK_STATE["prepared"]
            assert prepared is not None, "Apply preprocessing first"
            fits = flag_fit_quality(fit_peak_series(prepared, _plan_from_controls(fit_controls.get_params()), intensity_var="intensity_band_normalized", q_range=(fit_controls.q_min.value, fit_controls.q_max.value), pattern_indices=np.arange(min(batch_limit.value, prepared.sizes["pattern"]))), max_center_error=0.02)
            lattice = add_lattice_results(fits, hkls=((1, 1, 1),))
            calibration = LinearThermalExpansion(float(lattice.lattice_mean_A.isel(fit_pattern=0)), 300.0, 9e-6)
            time_coordinate = _thermal_time_coordinate(lattice)
            thermal = add_temperature_results(lattice, calibration, time_coord=time_coordinate)
            NOTEBOOK_STATE.update(fits=fits, thermal=thermal, thermal_time_coord=time_coordinate)
            display(plot_thermal_history(thermal))
            status.value = f"<b>Saved {thermal.sizes['fit_pattern']} bounded fit rows using {time_coordinate} for K/s.</b>"
        except Exception as exc:
            status.value = f"<b>Batch fit failed:</b> {exc}"
            raise

def export_compact(_=None):
    thermal = NOTEBOOK_STATE["thermal"]
    assert thermal is not None, "Run bounded batch before exporting"
    paths = export_time_resolved_results(thermal, netcdf_path=export_directory / "time_resolved_results.nc", csv_path=export_directory / "time_resolved_scalars.csv")
    status.value = f"<b>Exported compact results to {paths['netcdf'].parent}.</b>"
    return paths

discover_button.on_click(discover_folder_scans); load_button.on_click(load_processed); preprocess_button.on_click(apply_preprocessing); inspect_button.on_click(inspect_lazy_row); pilot_button.on_click(run_pilot); batch_button.on_click(run_bounded_batch); export_button.on_click(export_compact)
frame_row.observe(lambda change: inspect_lazy_row() if NOTEBOOK_STATE["series"] is not None else None, names="value")
NOTEBOOK_ACTIONS = {"discover_folder_scans": discover_folder_scans, "load_processed": load_processed, "apply_preprocessing": apply_preprocessing, "inspect_lazy_row": inspect_lazy_row, "run_pilot": run_pilot, "run_bounded_batch": run_bounded_batch, "export_compact": export_compact}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    load_processed(); apply_preprocessing(); inspect_lazy_row(); run_pilot(); run_bounded_batch()
